# 🏥 Fine-tuning PhoBERT cho Nhận Diện Thực Thể Y Tế (Medical NER) - MediBot AI

Notebook này được thiết kế để huấn luyện mô hình **PhoBERT** (`vinai/phobert-base` hoặc `vinai/phobert-large`) trên **Google Colab GPU (T4 / V100 / A100)**.

### Bộ nhãn thực thể y tế (BIO Tags):
- `B-SYMPTOM`, `I-SYMPTOM`: Triệu chứng cơ năng & thực thể (sốt cao, đau đầu, khó thở...)
- `B-VITAL`, `I-VITAL`: Chỉ số sinh tồn (39 độ, 140/90 mmHg, SpO2 92%...)
- `B-DURATION`, `I-DURATION`: Thời gian khởi phát & diễn biến (2 ngày nay, 3 tuần...)
- `B-DRUG`, `I-DRUG`: Thuốc & dược phẩm (Paracetamol, Oresol, Aspirin...)
- `B-RED_FLAG`, `I-RED_FLAG`: Dấu hiệu cảnh báo nguy kịch (chảy máu chân răng, nôn ra máu, méo miệng...)
- `O`: Các từ thông thường khác

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q transformers datasets seqeval accelerate pyvi

In [ ]:
# 3. Khai báo danh mục nhãn BIO
LABEL_LIST = [
    "O",
    "B-SYMPTOM", "I-SYMPTOM",
    "B-VITAL", "I-VITAL",
    "B-DURATION", "I-DURATION",
    "B-DRUG", "I-DRUG",
    "B-RED_FLAG", "I-RED_FLAG"
]
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}
print(f"Số lượng nhãn thực thể: {len(LABEL_LIST)}")

In [ ]:
# 4. Tải Tokenizer và Mô hình PhoBERT
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_NAME = "vinai/phobert-base"  # Có thể đổi sang "vinai/phobert-large" nếu dùng GPU A100
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
print("Đã nạp thành công mô hình:", MODEL_NAME)

In [ ]:
# 5. Chuẩn bị tập dữ liệu huấn luyện (Huấn luyện mẫu 10 câu lâm sàng)
raw_dataset = [
    {
        "tokens": ["Tôi", "bị", "sốt", "cao", "39", "độ", "từ", "2", "ngày", "nay"],
        "tags": ["O", "O", "B-SYMPTOM", "I-SYMPTOM", "B-VITAL", "I-VITAL", "O", "B-DURATION", "I-DURATION", "I-DURATION"]
    },
    {
        "tokens": ["Bệnh", "nhân", "đau", "thắt", "ngực", "dữ", "dội", "kèm", "vã", "mồ", "hôi", "lạnh"],
        "tags": ["O", "O", "B-RED_FLAG", "I-RED_FLAG", "I-RED_FLAG", "I-RED_FLAG", "I-RED_FLAG", "O", "B-RED_FLAG", "I-RED_FLAG", "I-RED_FLAG", "I-RED_FLAG"]
    },
    {
        "tokens": ["Bác", "sĩ", "kê", "Paracetamol", "500mg", "uống", "ngày", "2", "viên"],
        "tags": ["O", "O", "O", "B-DRUG", "I-DRUG", "O", "O", "O", "O"]
    },
    {
        "tokens": ["Tôi", "thấy", "chảy", "máu", "chân", "răng", "và", "xuất", "huyết", "dưới", "da"],
        "tags": ["O", "O", "B-RED_FLAG", "I-RED_FLAG", "I-RED_FLAG", "I-RED_FLAG", "O", "B-RED_FLAG", "I-RED_FLAG", "I-RED_FLAG", "I-RED_FLAG"]
    },
    {
        "tokens": ["Huyết", "áp", "đo", "được", "160/95", "mmHg", "đau", "đầu", "vùng", "gáy"],
        "tags": ["B-VITAL", "I-VITAL", "O", "O", "B-VITAL", "I-VITAL", "B-SYMPTOM", "I-SYMPTOM", "O", "O"]
    }
]
print(f"Tập dữ liệu mẫu gồm: {len(raw_dataset)} mẫu")

In [ ]:
# 6. Cấu hình TrainingArguments và Huấn luyện
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

training_args = TrainingArguments(
    output_dir="./phobert_medical_ner_output",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    save_total_limit=2,
    report_to="none"
)
print("Cấu hình Trainer đã sẵn sàng.")

In [ ]:
# 7. Thử nghiệm suy luận (Inference Demo)
from transformers import pipeline

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")
test_sentence = "Tôi bị sốt cao 39 độ và chảy máu chân răng từ sáng nay"
results = ner_pipeline(test_sentence)
print("Kết quả bóc tách thực thể:")
for r in results:
    print(f"- {r['word']}: {r['entity_group']} (Độ tin cậy: {r['score']:.4f})")

In [ ]:
# 8. Xuất mô hình về file zip để tải về đưa vào MediBot AI
model.save_pretrained("./phobert_ner_final")
tokenizer.save_pretrained("./phobert_ner_final")
!zip -r phobert_ner_final.zip ./phobert_ner_final
print("Đã nén file phobert_ner_final.zip thành công. Hãy tải về máy và đặt vào thư mục apps/ai_engine/models_weights/phobert_ner/")